# SSP Regridding y Bias Correction - ACCESS-CM2

Este notebook implementa el regridding y bias correction para archivos SSP ya concatenados y con coordenadas corregidas.

## Región: Valle de Aconcagua, Chile
- **Variables**: pr, tasmin, tasmax
- **Escenarios**: SSP245, SSP370, SSP585
- **Período**: 2015-2100
- **Método**: Regridding + Detrended Quantile Mapping (DQM)
- **Resolución final**: 0.05° (rejilla CR2MET)

## Pipeline de Procesamiento:
1. **Cargar archivos concatenados** con coordenadas corregidas
2. **Regridding espacial** CMIP6 → CR2MET (0.05°)
3. **Recorte espacial** a Valle de Aconcagua
4. **Cargar parámetros** de bias correction del historical
5. **Aplicar bias correction** usando DQM pre-entrenado
6. **Exportar NetCDF** finales corregidos

## Archivos de entrada:
```
out/ssp_concatenated/ACCESS-CM2/
├── pr_ACCESS-CM2_ssp245_concatenated_coords_2015-2100.nc
├── pr_ACCESS-CM2_ssp370_concatenated_coords_2015-2100.nc
├── pr_ACCESS-CM2_ssp585_concatenated_coords_2015-2100.nc
├── tasmax_ACCESS-CM2_ssp245_concatenated_coords_2015-2100.nc
├── tasmax_ACCESS-CM2_ssp370_concatenated_coords_2015-2100.nc
├── tasmax_ACCESS-CM2_ssp585_concatenated_coords_2015-2100.nc
├── tasmin_ACCESS-CM2_ssp245_concatenated_coords_2015-2100.nc
├── tasmin_ACCESS-CM2_ssp370_concatenated_coords_2015-2100.nc
└── tasmin_ACCESS-CM2_ssp585_concatenated_coords_2015-2100.nc
```

In [10]:
# ============================================================
# 🚀 SECCIÓN 1: SETUP Y CONFIGURACIÓN
# ============================================================

import xarray as xr
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import time
from datetime import datetime
import sys
import pickle
import json
import logging
import rioxarray  # Alternativa para regridding
from scipy.interpolate import griddata

# Importación condicional de xESMF
try:
    import xesmf as xe
    XESMF_AVAILABLE = True
    print(f"xesmf version: {xe.__version__}")
except ImportError:
    XESMF_AVAILABLE = False
    print("⚠️ xESMF no disponible, usando scipy para regridding")

# Configurar warnings
warnings.filterwarnings('ignore')

# Configurar logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("=" * 70)
print("🌍 SSP REGRIDDING & BIAS CORRECTION PIPELINE - ACCESS-CM2")
print("=" * 70)
print(f"Inicio: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"xarray version: {xr.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")

# Verificar si podemos hacer regridding
can_regrid = True
print(f"🔧 Capacidad de regridding: {'✅' if can_regrid else '❌'}")

xesmf version: 0.8.10
🌍 SSP REGRIDDING & BIAS CORRECTION PIPELINE - ACCESS-CM2
Inicio: 2025-10-12 22:18:51
xarray version: 2025.1.2
pandas version: 2.3.3
numpy version: 2.3.3
🔧 Capacidad de regridding: ✅


In [22]:
# ============================================================
# 📁 CONFIGURACIÓN DE RUTAS Y PARÁMETROS
# ============================================================

# Definir rutas principales
base_dir = Path("/home/aninotna/magister/tesis/justh2_pipeline")
data_dir = base_dir / "data"
cr2met_path = data_dir / "cr2met/clima.zarr"

# Directorios de entrada y salida
out_dir = base_dir / "out"
concatenated_dir = out_dir / "ssp_concatenated/ACCESS-CM2"  # Archivos concatenados de entrada
bias_params_dir = out_dir / "bias_params/ACCESS-CM2"        # Parámetros de bias correction
regridded_dir = out_dir / "ssp_regridded/ACCESS-CM2"        # Salida regridding
final_corrected_dir = out_dir / "ssp_bias_corrected/ACCESS-CM2"  # Salida final
logs_dir = base_dir / "logs"

# Crear directorios de salida
for directory in [regridded_dir, final_corrected_dir, logs_dir]:
    directory.mkdir(parents=True, exist_ok=True)

print("📁 CONFIGURACIÓN DE RUTAS:")
print(f"  • Base: {base_dir}")
print(f"  • CR2MET: {cr2met_path}")
print(f"  • Concatenated SSP (entrada): {concatenated_dir}")
print(f"  • Bias params: {bias_params_dir}")
print(f"  • Regridded output: {regridded_dir}")
print(f"  • Final corrected output: {final_corrected_dir}")
print(f"  • Logs: {logs_dir}")

# Verificar que los directorios críticos existen
print(f"\n🔍 VERIFICACIÓN:")
print(f"  CR2MET existe: {'✅' if cr2met_path.exists() else '❌'}")
print(f"  Concatenated SSP existe: {'✅' if concatenated_dir.exists() else '❌'}")
print(f"  Bias params existe: {'✅' if bias_params_dir.exists() else '❌'}")

# Contar archivos de entrada
if concatenated_dir.exists():
    input_files = list(concatenated_dir.glob("*_concatenated_coords_*.nc"))
    print(f"  Archivos SSP concatenados: {len(input_files)}")
    if len(input_files) != 9:
        print(f"  ⚠️ Se esperaban 9 archivos, encontrados: {len(input_files)}")
else:
    print(f"  ❌ Directorio de archivos concatenados no existe")

if not bias_params_dir.exists():
    print(f"\n⚠️ ADVERTENCIA: No se encontraron parámetros de bias correction")
    print(f"   Asegúrate de haber ejecutado el notebook de historical bias correction")

📁 CONFIGURACIÓN DE RUTAS:
  • Base: /home/aninotna/magister/tesis/justh2_pipeline
  • CR2MET: /home/aninotna/magister/tesis/justh2_pipeline/data/cr2met/clima.zarr
  • Concatenated SSP (entrada): /home/aninotna/magister/tesis/justh2_pipeline/out/ssp_concatenated/ACCESS-CM2
  • Bias params: /home/aninotna/magister/tesis/justh2_pipeline/out/bias_params/ACCESS-CM2
  • Regridded output: /home/aninotna/magister/tesis/justh2_pipeline/out/ssp_regridded/ACCESS-CM2
  • Final corrected output: /home/aninotna/magister/tesis/justh2_pipeline/out/ssp_bias_corrected/ACCESS-CM2
  • Logs: /home/aninotna/magister/tesis/justh2_pipeline/logs

🔍 VERIFICACIÓN:
  CR2MET existe: ✅
  Concatenated SSP existe: ✅
  Bias params existe: ✅
  Archivos SSP concatenados: 9


In [23]:
# ============================================================
# ⚙️ CONFIGURACIÓN DE PARÁMETROS DEL PROCESAMIENTO
# ============================================================

# Configuración de escenarios y variables
ssp_scenarios = ['ssp245', 'ssp370', 'ssp585']
variables = ['pr', 'tasmin', 'tasmax']
model_name = 'ACCESS-CM2'

# Configuración región Valle de Aconcagua
valle_aconcagua_bounds = {
    'lat_min': -33.27,
    'lat_max': -32.26,
    'lon_min': -71.89,
    'lon_max': -70.00
}

# Configuración de regridding
regridding_config = {
    'target_resolution': 0.05,  # Resolución CR2MET
    'method': 'bilinear',       # Método de interpolación
    'chunk_size': {'time': 1000, 'lat': 50, 'lon': 50},  # Para optimizar memoria
    'reuse_weights': True       # Reutilizar pesos de regridding
}

# Configuración de variables de temperatura (mapeo de nombres)
variable_mapping = {
    'pr': 'pr',
    'tasmin': 'tmin',  # Para bias correction se llama tmin
    'tasmax': 'tmax'   # Para bias correction se llama tmax
}

print("⚙️ CONFIGURACIÓN DEL PROCESAMIENTO:")
print(f"📊 Modelo: {model_name}")
print(f"🔮 Escenarios: {ssp_scenarios}")
print(f"📈 Variables: {variables}")

print(f"\n📍 Región Valle de Aconcagua:")
for key, value in valle_aconcagua_bounds.items():
    print(f"    {key}: {value}")

print(f"\n🌍 Configuración de regridding:")
for key, value in regridding_config.items():
    print(f"  {key}: {value}")

print(f"\n🔄 Mapeo de variables para bias correction:")
for ssp_var, bias_var in variable_mapping.items():
    print(f"  {ssp_var} → {bias_var}")

⚙️ CONFIGURACIÓN DEL PROCESAMIENTO:
📊 Modelo: ACCESS-CM2
🔮 Escenarios: ['ssp245', 'ssp370', 'ssp585']
📈 Variables: ['pr', 'tasmin', 'tasmax']

📍 Región Valle de Aconcagua:
    lat_min: -33.27
    lat_max: -32.26
    lon_min: -71.89
    lon_max: -70.0

🌍 Configuración de regridding:
  target_resolution: 0.05
  method: bilinear
  chunk_size: {'time': 1000, 'lat': 50, 'lon': 50}
  reuse_weights: True

🔄 Mapeo de variables para bias correction:
  pr → pr
  tasmin → tmin
  tasmax → tmax


## 🔍 SECCIÓN 2: CARGA Y VERIFICACIÓN DE DATOS

Antes de proceder con el regridding, necesitamos:
1. **Cargar CR2MET** como rejilla de referencia
2. **Verificar archivos SSP** concatenados disponibles
3. **Inspeccionar compatibilidad** espacial y temporal
4. **Verificar parámetros** de bias correction disponibles

In [24]:
# ============================================================
# 🌐 CARGAR CR2MET COMO REJILLA DE REFERENCIA
# ============================================================

def load_cr2met_reference():
    """Carga CR2MET como rejilla de referencia para el regridding"""
    
    print(f"\n{'='*70}")
    print(f"🌐 CARGANDO CR2MET COMO REJILLA DE REFERENCIA")
    print(f"{'='*70}")
    
    try:
        # Cargar CR2MET
        print(f"📂 Cargando CR2MET desde: {cr2met_path}")
        cr2met = xr.open_zarr(cr2met_path)
        
        # Mostrar información general
        print(f"\n📊 INFORMACIÓN CR2MET:")
        print(f"  📈 Variables: {list(cr2met.data_vars)}")
        print(f"  📊 Dimensiones: {dict(cr2met.dims)}")
        
        # Información espacial
        lat_cr2met = cr2met.lat.values
        lon_cr2met = cr2met.lon.values
        
        print(f"\n🌍 INFORMACIÓN ESPACIAL:")
        print(f"  📍 Latitudes: {lat_cr2met.min():.3f} a {lat_cr2met.max():.3f} ({len(lat_cr2met)} puntos)")
        print(f"  📍 Longitudes: {lon_cr2met.min():.3f} a {lon_cr2met.max():.3f} ({len(lon_cr2met)} puntos)")
        print(f"  📐 Resolución: ~{abs(lat_cr2met[1] - lat_cr2met[0]):.3f}°")
        
        # Información temporal
        cr2met_time = cr2met.time
        print(f"\n📅 INFORMACIÓN TEMPORAL:")
        print(f"  📅 Período: {pd.to_datetime(cr2met_time.values[0]).strftime('%Y-%m-%d')} a {pd.to_datetime(cr2met_time.values[-1]).strftime('%Y-%m-%d')}")
        print(f"  📈 Total días: {len(cr2met_time):,}")
        
        # Verificar cobertura Valle de Aconcagua
        valle_lat_covered = (lat_cr2met.min() <= valle_aconcagua_bounds['lat_min'] and 
                           lat_cr2met.max() >= valle_aconcagua_bounds['lat_max'])
        valle_lon_covered = (lon_cr2met.min() <= valle_aconcagua_bounds['lon_min'] and 
                           lon_cr2met.max() >= valle_aconcagua_bounds['lon_max'])
        
        valle_status = "✅" if (valle_lat_covered and valle_lon_covered) else "⚠️"
        print(f"\n🎯 COBERTURA VALLE DE ACONCAGUA: {valle_status}")
        
        if valle_lat_covered and valle_lon_covered:
            print(f"  ✅ CR2MET cubre completamente la región de interés")
            
            # Calcular índices para recorte posterior
            lat_mask = (lat_cr2met >= valle_aconcagua_bounds['lat_min']) & (lat_cr2met <= valle_aconcagua_bounds['lat_max'])
            lon_mask = (lon_cr2met >= valle_aconcagua_bounds['lon_min']) & (lon_cr2met <= valle_aconcagua_bounds['lon_max'])
            
            valle_points_lat = np.sum(lat_mask)
            valle_points_lon = np.sum(lon_mask)
            
            print(f"  📊 Puntos en Valle de Aconcagua: {valle_points_lat} × {valle_points_lon} = {valle_points_lat * valle_points_lon}")
        else:
            print(f"  ⚠️ CR2MET no cubre completamente la región")
        
        print(f"\n✅ CR2MET cargado exitosamente como rejilla de referencia")
        
        return cr2met
        
    except Exception as e:
        print(f"❌ Error al cargar CR2MET: {e}")
        return None

# Cargar CR2MET
cr2met_ref = load_cr2met_reference()

if cr2met_ref is not None:
    print(f"\n🎯 CR2MET listo para usar como rejilla objetivo del regridding")
else:
    print(f"\n❌ No se pudo cargar CR2MET. Revisa la ruta y el archivo.")


🌐 CARGANDO CR2MET COMO REJILLA DE REFERENCIA
📂 Cargando CR2MET desde: /home/aninotna/magister/tesis/justh2_pipeline/data/cr2met/clima.zarr

📊 INFORMACIÓN CR2MET:
  📈 Variables: ['year', 'cl_mask', 'pr', 'tmin', 'pr_sd', 'tmax']
  📊 Dimensiones: {'time': 22646, 'lat': 800, 'lon': 220}

🌍 INFORMACIÓN ESPACIAL:
  📍 Latitudes: -56.975 a -17.025 (800 puntos)
  📍 Longitudes: -76.975 a -66.025 (220 puntos)
  📐 Resolución: ~0.050°

📅 INFORMACIÓN TEMPORAL:
  📅 Período: 1960-01-01 a 2021-12-31
  📈 Total días: 22,646

🎯 COBERTURA VALLE DE ACONCAGUA: ✅
  ✅ CR2MET cubre completamente la región de interés
  📊 Puntos en Valle de Aconcagua: 20 × 38 = 760

✅ CR2MET cargado exitosamente como rejilla de referencia

🎯 CR2MET listo para usar como rejilla objetivo del regridding


In [25]:
# ============================================================
# 📂 DESCUBRIR Y VERIFICAR ARCHIVOS SSP CONCATENADOS
# ============================================================

def discover_concatenated_ssp_files():
    """Descubre y organiza los archivos SSP concatenados disponibles"""
    
    print(f"\n{'='*70}")
    print(f"📂 DESCUBRIENDO ARCHIVOS SSP CONCATENADOS")
    print(f"{'='*70}")
    
    available_files = {}
    
    if not concatenated_dir.exists():
        print(f"❌ Directorio no existe: {concatenated_dir}")
        return available_files
    
    # Buscar archivos con el patrón correcto
    pattern = "*_concatenated_coords_*.nc"
    all_files = list(concatenated_dir.glob(pattern))
    
    print(f"📁 Directorio: {concatenated_dir}")
    print(f"🔍 Patrón de búsqueda: {pattern}")
    print(f"📦 Archivos encontrados: {len(all_files)}")
    
    if not all_files:
        print(f"❌ No se encontraron archivos concatenados")
        print(f"💡 Ejecuta primero el notebook 01_ssp_preprocessing_and_bias_correction.ipynb")
        return available_files
    
    # Organizar archivos por variable y escenario
    for file_path in all_files:
        file_name = file_path.name
        print(f"\n📄 {file_name}")
        
        # Extraer variable y escenario del nombre
        # Formato esperado: {var}_{model}_{scenario}_concatenated_coords_2015-2100.nc
        parts = file_name.split('_')
        
        if len(parts) >= 3:
            var = parts[0]
            scenario = parts[2]
            
            if var in variables and scenario in ssp_scenarios:
                if var not in available_files:
                    available_files[var] = {}
                
                available_files[var][scenario] = file_path
                
                # Verificar información básica del archivo
                try:
                    with xr.open_dataset(file_path) as ds:
                        file_size_mb = file_path.stat().st_size / (1024 * 1024)
                        
                        print(f"  ✅ Variable: {var}, Escenario: {scenario}")
                        print(f"  📊 Dimensiones: {dict(ds.dims)}")
                        print(f"  📁 Tamaño: {file_size_mb:.1f} MB")
                        print(f"  📅 Período: {ds.time.min().dt.strftime('%Y-%m-%d').values} a {ds.time.max().dt.strftime('%Y-%m-%d').values}")
                        
                        # Verificar coordenadas
                        lon_range = f"{ds.lon.min().values:.1f} a {ds.lon.max().values:.1f}"
                        print(f"  🌍 Longitudes: {lon_range} {'✅' if ds.lon.max().values <= 180 else '⚠️'}")
                        
                except Exception as e:
                    print(f"  ❌ Error al inspeccionar: {e}")
                    
            else:
                print(f"  ⚠️ Variable o escenario no reconocido: {var}, {scenario}")
        else:
            print(f"  ⚠️ Formato de nombre no reconocido")
    
    # Resumen de disponibilidad
    print(f"\n{'='*50}")
    print(f"📊 RESUMEN DE DISPONIBILIDAD")
    print(f"{'='*50}")
    
    total_expected = len(variables) * len(ssp_scenarios)
    total_found = sum(len(scenarios) for scenarios in available_files.values())
    
    print(f"📈 Esperados: {total_expected} archivos ({len(variables)} variables × {len(ssp_scenarios)} escenarios)")
    print(f"📦 Encontrados: {total_found} archivos")
    
    if total_found == total_expected:
        print(f"✅ COMPLETO: Todos los archivos están disponibles")
    else:
        print(f"⚠️ INCOMPLETO: Faltan {total_expected - total_found} archivos")
    
    # Mostrar matriz de disponibilidad
    print(f"\n📋 MATRIZ DE DISPONIBILIDAD:")
    print(f"{'Variable':<10} | {'SSP245':<8} | {'SSP370':<8} | {'SSP585':<8}")
    print(f"{"-"*10} | {"-"*8} | {"-"*8} | {"-"*8}")
    
    for var in variables:
        row = f"{var:<10} |"
        for scenario in ssp_scenarios:
            status = "✅" if var in available_files and scenario in available_files[var] else "❌"
            row += f" {status:<8} |"
        print(row)
    
    return available_files

# Descubrir archivos
ssp_files = discover_concatenated_ssp_files()

if ssp_files:
    print(f"\n🎯 Archivos SSP listos para regridding")
else:
    print(f"\n❌ No se encontraron archivos SSP concatenados")


📂 DESCUBRIENDO ARCHIVOS SSP CONCATENADOS
📁 Directorio: /home/aninotna/magister/tesis/justh2_pipeline/out/ssp_concatenated/ACCESS-CM2
🔍 Patrón de búsqueda: *_concatenated_coords_*.nc
📦 Archivos encontrados: 9

📄 pr_ACCESS-CM2_ssp245_concatenated_coords_2015-2100.nc
  ✅ Variable: pr, Escenario: ssp245
  📊 Dimensiones: {'time': 31411, 'bnds': 2, 'lat': 144, 'lon': 192}
  📁 Tamaño: 2751.6 MB
  📅 Período: 2015-01-01 a 2100-12-31
  🌍 Longitudes: -179.1 a 179.1 ✅

📄 pr_ACCESS-CM2_ssp370_concatenated_coords_2015-2100.nc
  ✅ Variable: pr, Escenario: ssp370
  📊 Dimensiones: {'time': 31411, 'bnds': 2, 'lat': 144, 'lon': 192}
  📁 Tamaño: 2751.5 MB
  📅 Período: 2015-01-01 a 2100-12-31
  🌍 Longitudes: -179.1 a 179.1 ✅

📄 pr_ACCESS-CM2_ssp585_concatenated_coords_2015-2100.nc
  ✅ Variable: pr, Escenario: ssp585
  📊 Dimensiones: {'time': 31411, 'bnds': 2, 'lat': 144, 'lon': 192}
  📁 Tamaño: 2751.1 MB
  📅 Período: 2015-01-01 a 2100-12-31
  🌍 Longitudes: -179.1 a 179.1 ✅

📄 tasmin_ACCESS-CM2_ssp245_conc

## 🌍 SECCIÓN 3: REGRIDDING ESPACIAL

Con los archivos verificados, procederemos con el regridding de CMIP6 nativo (~2.5°) a resolución CR2MET (0.05°):

1. **Crear rejilla objetivo** basada en CR2MET
2. **Función de regridding** usando interpolación bilinear
3. **Procesamiento por lotes** para optimizar memoria
4. **Recorte espacial** a Valle de Aconcagua
5. **Export NetCDF** regridded

In [26]:
# ============================================================
# 🎯 CREAR REJILLA OBJETIVO BASADA EN CR2MET
# ============================================================

def create_target_grid_from_cr2met():
    """
    Crea la rejilla objetivo para regridding basada en CR2MET
    con recorte opcional a Valle de Aconcagua
    """
    
    print(f"\n{'='*70}")
    print(f"🎯 CREANDO REJILLA OBJETIVO BASADA EN CR2MET")
    print(f"{'='*70}")
    
    if cr2met_ref is None:
        print(f"❌ CR2MET no está cargado")
        return None
    
    try:
        # Extraer coordenadas de CR2MET
        lat_cr2met = cr2met_ref.lat.values
        lon_cr2met = cr2met_ref.lon.values
        
        print(f"📊 REJILLA CR2MET ORIGINAL:")
        print(f"  📍 Latitudes: {lat_cr2met.min():.3f} a {lat_cr2met.max():.3f} ({len(lat_cr2met)} puntos)")
        print(f"  📍 Longitudes: {lon_cr2met.min():.3f} a {lon_cr2met.max():.3f} ({len(lon_cr2met)} puntos)")
        print(f"  📐 Resolución: ~{abs(lat_cr2met[1] - lat_cr2met[0]):.3f}°")
        
        # Opción 1: Rejilla completa CR2MET
        target_grid_full = xr.Dataset({
            'lat': (['lat'], lat_cr2met),
            'lon': (['lon'], lon_cr2met)
        })
        
        # Opción 2: Rejilla recortada a Valle de Aconcagua (más eficiente)
        lat_mask = ((lat_cr2met >= valle_aconcagua_bounds['lat_min'] - 0.1) & 
                    (lat_cr2met <= valle_aconcagua_bounds['lat_max'] + 0.1))
        lon_mask = ((lon_cr2met >= valle_aconcagua_bounds['lon_min'] - 0.1) & 
                    (lon_cr2met <= valle_aconcagua_bounds['lon_max'] + 0.1))
        
        lat_valle = lat_cr2met[lat_mask]
        lon_valle = lon_cr2met[lon_mask]
        
        target_grid_valle = xr.Dataset({
            'lat': (['lat'], lat_valle),
            'lon': (['lon'], lon_valle)
        })
        
        print(f"\n🎯 REJILLA OBJETIVO - VALLE DE ACONCAGUA:")
        print(f"  📍 Latitudes: {lat_valle.min():.3f} a {lat_valle.max():.3f} ({len(lat_valle)} puntos)")
        print(f"  📍 Longitudes: {lon_valle.min():.3f} a {lon_valle.max():.3f} ({len(lon_valle)} puntos)")
        print(f"  📊 Total puntos: {len(lat_valle) * len(lon_valle):,}")
        print(f"  📐 Resolución: ~{abs(lat_valle[1] - lat_valle[0]):.3f}°")
        
        # Verificar cobertura
        valle_covered = (lat_valle.min() <= valle_aconcagua_bounds['lat_min'] and 
                        lat_valle.max() >= valle_aconcagua_bounds['lat_max'] and
                        lon_valle.min() <= valle_aconcagua_bounds['lon_min'] and 
                        lon_valle.max() >= valle_aconcagua_bounds['lon_max'])
        
        print(f"\n✅ Cobertura Valle de Aconcagua: {'COMPLETA' if valle_covered else 'PARCIAL'}")
        
        # Agregar metadatos
        for grid in [target_grid_full, target_grid_valle]:
            grid.lat.attrs.update({
                'long_name': 'latitude',
                'standard_name': 'latitude',
                'units': 'degrees_north',
                'axis': 'Y'
            })
            grid.lon.attrs.update({
                'long_name': 'longitude', 
                'standard_name': 'longitude',
                'units': 'degrees_east',
                'axis': 'X'
            })
        
        print(f"\n🎯 Rejillas objetivo creadas exitosamente")
        print(f"  📊 Rejilla completa: {len(lat_cr2met)} × {len(lon_cr2met)} puntos")
        print(f"  🎯 Rejilla Valle: {len(lat_valle)} × {len(lon_valle)} puntos")
        print(f"  💡 Usando rejilla Valle para eficiencia")
        
        return {
            'full': target_grid_full,
            'valle': target_grid_valle,
            'lat_mask': lat_mask,
            'lon_mask': lon_mask
        }
        
    except Exception as e:
        print(f"❌ Error al crear rejilla objetivo: {e}")
        return None

# Crear rejilla objetivo
target_grids = create_target_grid_from_cr2met()

if target_grids is not None:
    print(f"\n✅ Rejilla objetivo lista para regridding")
    target_grid = target_grids['valle']  # Usar rejilla recortada por defecto
else:
    print(f"\n❌ No se pudo crear rejilla objetivo")


🎯 CREANDO REJILLA OBJETIVO BASADA EN CR2MET
📊 REJILLA CR2MET ORIGINAL:
  📍 Latitudes: -56.975 a -17.025 (800 puntos)
  📍 Longitudes: -76.975 a -66.025 (220 puntos)
  📐 Resolución: ~0.050°

🎯 REJILLA OBJETIVO - VALLE DE ACONCAGUA:
  📍 Latitudes: -33.325 a -32.175 (24 puntos)
  📍 Longitudes: -71.975 a -69.925 (42 puntos)
  📊 Total puntos: 1,008
  📐 Resolución: ~0.050°

✅ Cobertura Valle de Aconcagua: COMPLETA

🎯 Rejillas objetivo creadas exitosamente
  📊 Rejilla completa: 800 × 220 puntos
  🎯 Rejilla Valle: 24 × 42 puntos
  💡 Usando rejilla Valle para eficiencia

✅ Rejilla objetivo lista para regridding


In [27]:
# ============================================================
# 🌍 FUNCIÓN PRINCIPAL DE REGRIDDING
# ============================================================

def regrid_dataset_to_cr2met(ds_source, target_grid, method='bilinear', use_xesmf=True):
    """
    Realiza regridding de un dataset CMIP6 a la rejilla CR2MET
    
    Parameters:
    -----------
    ds_source : xr.Dataset
        Dataset fuente con resolución CMIP6
    target_grid : xr.Dataset
        Rejilla objetivo (CR2MET)
    method : str
        Método de interpolación ('bilinear', 'nearest', 'cubic')
    use_xesmf : bool
        Si True, usa xESMF; si False, usa scipy
    
    Returns:
    --------
    xr.Dataset
        Dataset regridded a resolución CR2MET
    """
    
    print(f"  🌍 Iniciando regridding...")
    print(f"    Método: {method}")
    
    # Verificar disponibilidad de xESMF
    if use_xesmf and not XESMF_AVAILABLE:
        print(f"    ⚠️ xESMF no disponible, usando scipy")
        use_xesmf = False
    
    print(f"    Motor: {'xESMF' if use_xesmf else 'scipy'}")
    
    try:
        # Información de rejillas
        source_shape = (len(ds_source.lat), len(ds_source.lon))
        target_shape = (len(target_grid.lat), len(target_grid.lon))
        
        print(f"    📊 Origen: {source_shape[0]} × {source_shape[1]} puntos")
        print(f"    📊 Destino: {target_shape[0]} × {target_shape[1]} puntos")
        
        # Detectar variable principal
        data_vars = [var for var in ds_source.data_vars if var not in ['time_bnds', 'lat_bnds', 'lon_bnds']]
        if not data_vars:
            raise ValueError("No se encontró variable de datos principal")
        
        main_var = data_vars[0]
        print(f"    📈 Variable: {main_var}")
        
        # Método 1: xESMF (recomendado)
        if use_xesmf:
            try:
                print(f"    🔧 Usando xESMF para regridding...")
                
                # Crear regridder sin reuse_weights para evitar errores
                regridder = xe.Regridder(
                    ds_source, 
                    target_grid,
                    method=method
                )
                
                # Aplicar regridding
                ds_regridded = regridder(ds_source)
                
                print(f"    ✅ Regridding xESMF exitoso")
                
            except Exception as e:
                print(f"    ⚠️ Error con xESMF: {e}")
                print(f"    🔄 Cambiando a método scipy...")
                use_xesmf = False
        
        # Método 2: scipy (alternativo)
        if not use_xesmf:
            print(f"    🔧 Usando scipy para regridding...")
            
            # Preparar coordenadas fuente
            lon_source, lat_source = np.meshgrid(ds_source.lon.values, ds_source.lat.values)
            points_source = np.column_stack([lon_source.ravel(), lat_source.ravel()])
            
            # Preparar coordenadas objetivo
            lon_target, lat_target = np.meshgrid(target_grid.lon.values, target_grid.lat.values)
            points_target = np.column_stack([lon_target.ravel(), lat_target.ravel()])
            
            # Mapear método scipy
            scipy_method = 'linear' if method == 'bilinear' else method
            if scipy_method == 'cubic':
                scipy_method = 'linear'  # cubic no disponible en griddata
            
            print(f"    📊 Puntos origen: {len(points_source):,}")
            print(f"    📊 Puntos destino: {len(points_target):,}")
            
            # Procesar por chunks temporales para ahorrar memoria
            time_chunks = regridding_config['chunk_size']['time']
            n_times = len(ds_source.time)
            
            regridded_data = []
            
            for i in range(0, n_times, time_chunks):
                end_i = min(i + time_chunks, n_times)
                chunk_times = slice(i, end_i)
                
                print(f"    🔄 Procesando chunk temporal {i+1}-{end_i} de {n_times}")
                
                # Extraer datos del chunk
                data_chunk = ds_source[main_var].isel(time=chunk_times).values
                
                # Regridding por paso de tiempo
                chunk_regridded = []
                for t in range(data_chunk.shape[0]):
                    data_2d = data_chunk[t, :, :].ravel()
                    
                    # Interpolación
                    data_interp = griddata(
                        points_source, 
                        data_2d, 
                        points_target, 
                        method=scipy_method,
                        fill_value=np.nan
                    )
                    
                    # Reshape a rejilla objetivo
                    data_grid = data_interp.reshape(target_shape)
                    chunk_regridded.append(data_grid)
                
                regridded_data.extend(chunk_regridded)
            
            # Crear dataset regridded CON COORDENADAS EXPLÍCITAS
            regridded_array = np.array(regridded_data)
            
            # CORRECCIÓN PRINCIPAL: Crear coordenadas como variables explícitas
            coords_dict = {
                'time': (['time'], ds_source.time.values, ds_source.time.attrs),
                'lat': (['lat'], target_grid.lat.values, target_grid.lat.attrs),
                'lon': (['lon'], target_grid.lon.values, target_grid.lon.attrs)
            }
            
            # Crear dataset con coordenadas explícitas
            ds_regridded = xr.Dataset(
                data_vars={
                    main_var: (['time', 'lat', 'lon'], regridded_array, ds_source[main_var].attrs)
                },
                coords=coords_dict,
                attrs=ds_source.attrs.copy()
            )
            
            print(f"    ✅ Regridding scipy exitoso")
        
        # Verificar resultado
        print(f"    📊 Resultado: {dict(ds_regridded.dims)}")
        print(f"    📈 Variable: {main_var}")
        print(f"    🗂️ Coordenadas: {list(ds_regridded.coords)}")
        
        # Verificar que las coordenadas son variables
        coords_as_vars = [coord for coord in ds_regridded.coords]
        print(f"    ✅ Coordenadas como variables: {coords_as_vars}")
        
        # Agregar metadatos de regridding (CORREGIDO: tipos compatibles con NetCDF)
        ds_regridded.attrs.update({
            'regridding_method': method,
            'regridding_engine': 'xESMF' if use_xesmf else 'scipy',
            'regridding_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'source_resolution': f"{source_shape[0]}x{source_shape[1]}",
            'target_resolution': f"{target_shape[0]}x{target_shape[1]}",
            'target_grid_type': 'CR2MET_Valle_Aconcagua'
        })
        
        # Metadatos de variable (CORREGIDO: convertir bool a int para NetCDF)
        ds_regridded[main_var].attrs.update({
            'regridding_applied': int(1),  # True → 1 (entero compatible con NetCDF)
            'original_grid_shape': f"{source_shape[0]}x{source_shape[1]}",
            'regridded_grid_shape': f"{target_shape[0]}x{target_shape[1]}"
        })
        
        return ds_regridded
        
    except Exception as e:
        print(f"    ❌ Error en regridding: {e}")
        raise


# Test de regridding con un archivo pequeño
def test_regridding_function():
    """Test rápido de la función de regridding corregida"""
    
    print(f"\n{'='*70}")
    print(f"🧪 TEST DE FUNCIÓN DE REGRIDDING CORREGIDA")
    print(f"{'='*70}")
    
    if not ssp_files or target_grids is None:
        print(f"❌ No hay archivos SSP o rejilla objetivo disponibles")
        return False
    
    # Tomar el primer archivo disponible para test
    test_var = list(ssp_files.keys())[0]
    test_scenario = list(ssp_files[test_var].keys())[0]
    test_file = ssp_files[test_var][test_scenario]
    
    print(f"📄 Archivo de test: {test_file.name}")
    print(f"🔬 Variable: {test_var}, Escenario: {test_scenario}")
    
    try:
        # Cargar solo una muestra temporal pequeña
        with xr.open_dataset(test_file) as ds_full:
            # Tomar solo los primeros 10 días para test
            ds_sample = ds_full.isel(time=slice(0, 10))
            
            print(f"📊 Sample shape: {dict(ds_sample.dims)}")
            
            # Test de regridding
            start_time = time.time()
            ds_regridded = regrid_dataset_to_cr2met(
                ds_sample, 
                target_grids['valle'], 
                method='bilinear'
            )
            regrid_time = time.time() - start_time
            
            print(f"\n✅ TEST EXITOSO:")
            print(f"  ⏱️ Tiempo: {regrid_time:.1f} segundos")
            print(f"  📊 Forma original: {len(ds_sample.lat)} × {len(ds_sample.lon)}")
            print(f"  📊 Forma regridded: {len(ds_regridded.lat)} × {len(ds_regridded.lon)}")
            print(f"  📈 Reducción de puntos: {(len(ds_sample.lat) * len(ds_sample.lon)) / (len(ds_regridded.lat) * len(ds_regridded.lon)):.1f}x")
            
            # Verificar que las coordenadas están como variables
            coords_as_vars = [coord for coord in ds_regridded.coords if coord in ds_regridded.variables]
            print(f"  ✅ Coordenadas como variables: {coords_as_vars}")
            
            # Crear archivo temporal para verificar guardado
            test_output = f"/tmp/test_regridding_fixed_{test_var}_{test_scenario}.nc"
            
            # Encoding mejorado
            encoding = {
                'time': {'zlib': True, 'complevel': 4},
                'lat': {'zlib': True, 'complevel': 4}, 
                'lon': {'zlib': True, 'complevel': 4},
                test_var: {'zlib': True, 'complevel': 4, '_FillValue': np.nan}
            }
            
            ds_regridded.to_netcdf(test_output, encoding=encoding)
            
            # Verificar archivo guardado
            import os
            file_size = os.path.getsize(test_output) / (1024*1024)  # MB
            print(f"  📏 Tamaño archivo de test: {file_size:.1f} MB")
            
            # Reabrir y verificar estructura
            with xr.open_dataset(test_output) as ds_check:
                coord_vars = [v for v in ds_check.variables if v in ['lat', 'lon', 'time']]
                print(f"  ✅ Verificación: coordenadas guardadas como variables: {coord_vars}")
                
                if len(coord_vars) >= 3:
                    print(f"  🎯 TEST COMPLETAMENTE EXITOSO - Coordenadas correctas")
                    os.remove(test_output)  # Limpiar
                    return True
                else:
                    print(f"  ❌ TEST FALLIDO - Coordenadas incompletas")
                    return False
            
            # Verificar rango de datos
            var_name = test_var
            original_range = f"{float(ds_sample[var_name].min()):.2f} a {float(ds_sample[var_name].max()):.2f}"
            regridded_range = f"{float(ds_regridded[var_name].min()):.2f} a {float(ds_regridded[var_name].max()):.2f}"
            
            print(f"  📈 Rango original: {original_range}")
            print(f"  📈 Rango regridded: {regridded_range}")
            
            # Limpiar memoria
            ds_regridded.close()
            
            return True
            
    except Exception as e:
        print(f"❌ Test fallido: {e}")
        return False


# Test de regridding con un archivo pequeño - VERSIÓN ANTERIOR
def test_regridding_function_original():
    """Test rápido de la función de regridding"""
    
    print(f"\n{'='*70}")
    print(f"🧪 TEST DE FUNCIÓN DE REGRIDDING")
    print(f"{'='*70}")
    
    if not ssp_files or target_grids is None:
        print(f"❌ No hay archivos SSP o rejilla objetivo disponibles")
        return False
    
    # Tomar el primer archivo disponible para test
    test_var = list(ssp_files.keys())[0]
    test_scenario = list(ssp_files[test_var].keys())[0]
    test_file = ssp_files[test_var][test_scenario]
    
    print(f"📄 Archivo de test: {test_file.name}")
    print(f"🔬 Variable: {test_var}, Escenario: {test_scenario}")
    
    try:
        # Cargar solo una muestra temporal pequeña
        with xr.open_dataset(test_file) as ds_full:
            # Tomar solo los primeros 10 días para test
            ds_sample = ds_full.isel(time=slice(0, 10))
            
            print(f"📊 Sample shape: {dict(ds_sample.dims)}")
            
            # Test de regridding
            start_time = time.time()
            ds_regridded = regrid_dataset_to_cr2met(
                ds_sample, 
                target_grids['valle'], 
                method='bilinear'
            )
            regrid_time = time.time() - start_time
            
            print(f"\n✅ TEST EXITOSO:")
            print(f"  ⏱️ Tiempo: {regrid_time:.1f} segundos")
            print(f"  📊 Forma original: {len(ds_sample.lat)} × {len(ds_sample.lon)}")
            print(f"  📊 Forma regridded: {len(ds_regridded.lat)} × {len(ds_regridded.lon)}")
            print(f"  📈 Reducción de puntos: {(len(ds_sample.lat) * len(ds_sample.lon)) / (len(ds_regridded.lat) * len(ds_regridded.lon)):.1f}x")
            
            # Verificar rango de datos
            var_name = test_var
            original_range = f"{float(ds_sample[var_name].min()):.2f} a {float(ds_sample[var_name].max()):.2f}"
            regridded_range = f"{float(ds_regridded[var_name].min()):.2f} a {float(ds_regridded[var_name].max()):.2f}"
            
            print(f"  📈 Rango original: {original_range}")
            print(f"  📈 Rango regridded: {regridded_range}")
            
            # Limpiar memoria
            ds_regridded.close()
            
            return True
            
    except Exception as e:
        print(f"❌ Test fallido: {e}")
        return False

# Ejecutar test solo si las variables están definidas
if 'can_regrid' in locals() and can_regrid and 'ssp_files' in locals() and 'target_grids' in locals():
    test_success = test_regridding_function()
    if test_success:
        print(f"\n🎯 Función de regridding validada y lista para procesamiento masivo")
    else:
        print(f"\n❌ Test de regridding falló. Revisa la configuración.")
else:
    print(f"\n⚠️ Variables no definidas aún. Ejecuta las celdas anteriores primero.")


🧪 TEST DE FUNCIÓN DE REGRIDDING CORREGIDA
📄 Archivo de test: pr_ACCESS-CM2_ssp245_concatenated_coords_2015-2100.nc
🔬 Variable: pr, Escenario: ssp245
📊 Sample shape: {'time': 10, 'bnds': 2, 'lat': 144, 'lon': 192}
  🌍 Iniciando regridding...
    Método: bilinear
    Motor: xESMF
    📊 Origen: 144 × 192 puntos
    📊 Destino: 24 × 42 puntos
    📈 Variable: pr
    🔧 Usando xESMF para regridding...
    ✅ Regridding xESMF exitoso
    📊 Resultado: {'time': 10, 'lat': 24, 'lon': 42}
    📈 Variable: pr
    🗂️ Coordenadas: ['time', 'lat', 'lon']
    ✅ Coordenadas como variables: ['time', 'lat', 'lon']

✅ TEST EXITOSO:
  ⏱️ Tiempo: 1.5 segundos
  📊 Forma original: 144 × 192
  📊 Forma regridded: 24 × 42
  📈 Reducción de puntos: 27.4x
  ✅ Coordenadas como variables: ['time', 'lat', 'lon']
  📏 Tamaño archivo de test: 0.0 MB
  ✅ Verificación: coordenadas guardadas como variables: ['time', 'lat', 'lon']
  🎯 TEST COMPLETAMENTE EXITOSO - Coordenadas correctas

🎯 Función de regridding validada y lista p

In [28]:
# ============================================================
# 🚀 PIPELINE PRINCIPAL DE REGRIDDING MASIVO
# ============================================================

def process_ssp_regridding_pipeline():
    """
    Pipeline principal para regridding masivo de todos los archivos SSP
    """
    
    print(f"\n{'='*70}")
    print(f"🚀 PIPELINE DE REGRIDDING MASIVO - SSP → CR2MET")
    print(f"{'='*70}")
    
    total_files = sum(len(scenarios) for scenarios in ssp_files.values())
    print(f"📊 Total archivos a procesar: {total_files}")
    print(f"🎯 Resolución objetivo: {regridding_config['target_resolution']}°")
    print(f"🔧 Método: {regridding_config['method']}")
    print(f"⏱️ Inicio: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    results = {}
    processing_log = []
    successful = 0
    failed = 0
    total_processing_time = 0
    
    for var in variables:
        if var not in ssp_files:
            print(f"\n⚠️ Variable {var} no disponible")
            continue
            
        results[var] = {}
        
        for scenario in ssp_scenarios:
            if scenario not in ssp_files[var]:
                print(f"\n⚠️ {var} - {scenario} no disponible")
                continue
                
            combination = f"{var}-{scenario}"
            input_file = ssp_files[var][scenario]
            
            print(f"\n{'🌟'*60}")
            print(f"📋 PROCESANDO: {combination.upper()}")
            print(f"{'🌟'*60}")
            print(f"📄 Archivo: {input_file.name}")
            
            start_time = time.time()
            
            try:
                # 1. Cargar archivo fuente
                print(f"\n1️⃣ CARGANDO ARCHIVO FUENTE")
                with xr.open_dataset(input_file) as ds_source:
                    input_size_mb = input_file.stat().st_size / (1024 * 1024)
                    print(f"  📄 {input_file.name}")
                    print(f"  📊 Dimensiones: {dict(ds_source.dims)}")
                    print(f"  📁 Tamaño: {input_size_mb:.1f} MB")
                    print(f"  📅 Período: {ds_source.time.min().dt.strftime('%Y-%m-%d').values} a {ds_source.time.max().dt.strftime('%Y-%m-%d').values}")
                    
                    # 2. Aplicar regridding
                    print(f"\n2️⃣ REGRIDDING ESPACIAL")
                    ds_regridded = regrid_dataset_to_cr2met(
                        ds_source, 
                        target_grids['valle'],
                        method=regridding_config['method']
                    )
                    
                    # 3. Verificar cobertura Valle de Aconcagua
                    print(f"\n3️⃣ VERIFICACIÓN ESPACIAL")
                    lat_coverage = (ds_regridded.lat.min().values <= valle_aconcagua_bounds['lat_min'] and 
                                  ds_regridded.lat.max().values >= valle_aconcagua_bounds['lat_max'])
                    lon_coverage = (ds_regridded.lon.min().values <= valle_aconcagua_bounds['lon_min'] and 
                                  ds_regridded.lon.max().values >= valle_aconcagua_bounds['lon_max'])
                    
                    coverage_status = "✅" if (lat_coverage and lon_coverage) else "⚠️"
                    print(f"    Valle de Aconcagua cubierto: {coverage_status}")
                    print(f"    Latitudes: {ds_regridded.lat.min().values:.3f} a {ds_regridded.lat.max().values:.3f}")
                    print(f"    Longitudes: {ds_regridded.lon.min().values:.3f} a {ds_regridded.lon.max().values:.3f}")
                    
                    # 4. Agregar metadatos completos
                    print(f"\n4️⃣ METADATOS")
                    ds_regridded.attrs.update({
                        'title': f'SSP {scenario.upper()} regridded to CR2MET grid - {var}',
                        'model': model_name,
                        'scenario': scenario,
                        'variable': var,
                        'processing_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                        'processing_stage': 'regridded_to_CR2MET',
                        'target_region': 'Valle_de_Aconcagua_Chile',
                        'spatial_resolution': f'{regridding_config["target_resolution"]}°',
                        'temporal_resolution': 'daily',
                        'institution': 'Universidad_de_Chile',
                        'project': 'JUSTH2_Pipeline',
                        'source_file': input_file.name
                    })
                    
                    # Metadatos de variable
                    main_var = var
                    if main_var in ds_regridded.data_vars:
                        ds_regridded[main_var].attrs.update({
                            'processing_steps': 'concatenation,unit_conversion,coordinate_conversion,regridding',
                            'regridding_target': 'CR2MET_grid',
                            'regridding_method': regridding_config['method'],
                            'processing_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                        })
                    
                    # 5. Guardar archivo regridded
                    print(f"\n5️⃣ GUARDANDO ARCHIVO REGRIDDED")
                    output_filename = f"{var}_{model_name}_{scenario}_regridded_cr2met_2015-2100.nc"
                    output_path = regridded_dir / output_filename
                    
                    print(f"  📁 {output_filename}")
                    
                    # Guardar con compresión optimizada - ENCODING MEJORADO
                    encoding = {
                        'time': {'zlib': True, 'complevel': 4},
                        'lat': {'zlib': True, 'complevel': 4},
                        'lon': {'zlib': True, 'complevel': 4}
                    }
                    
                    # Agregar encoding para variables de datos
                    for var_name in ds_regridded.data_vars:
                        if var_name not in ['time_bnds']:
                            encoding[var_name] = {
                                'zlib': True, 
                                'complevel': 4, 
                                'shuffle': True,
                                '_FillValue': np.nan if ds_regridded[var_name].dtype.kind == 'f' else None
                            }
                    
                    ds_regridded.to_netcdf(output_path, encoding=encoding)
                    
                    # Verificar archivo guardado
                    output_size_mb = output_path.stat().st_size / (1024 * 1024)
                    compression_ratio = input_size_mb / output_size_mb if output_size_mb > 0 else 1
                    
                    print(f"  ✅ Archivo guardado exitosamente")
                    print(f"  📊 Tamaño: {output_size_mb:.1f} MB")
                    print(f"  📉 Compresión: {compression_ratio:.1f}x")
                    
                    # 6. Mostrar resumen
                    print(f"\n📋 RESUMEN FINAL:")
                    print(f"  📊 Regridding: {len(ds_source.lat)}×{len(ds_source.lon)} → {len(ds_regridded.lat)}×{len(ds_regridded.lon)}")
                    print(f"  📅 Período: {ds_regridded.time.min().dt.strftime('%Y-%m-%d').values} a {ds_regridded.time.max().dt.strftime('%Y-%m-%d').values}")
                    print(f"  🌍 Resolución: {regridding_config['target_resolution']}°")
                    print(f"  📈 Total días: {len(ds_regridded.time):,}")
                    
                    # Limpiar memoria
                    ds_regridded.close()
                
                # Registrar éxito
                processing_time = time.time() - start_time
                total_processing_time += processing_time
                
                results[var][scenario] = {
                    'status': 'success',
                    'input_file': input_file,
                    'output_file': output_path,
                    'input_size_mb': input_size_mb,
                    'output_size_mb': output_size_mb,
                    'compression_ratio': compression_ratio,
                    'processing_time': processing_time,
                    'valle_coverage': lat_coverage and lon_coverage
                }
                
                processing_log.append({
                    'combination': combination,
                    'status': 'SUCCESS',
                    'processing_time': processing_time,
                    'output_file': output_filename,
                    'output_size_mb': output_size_mb,
                    'compression_ratio': compression_ratio,
                    'valle_coverage': lat_coverage and lon_coverage
                })
                
                successful += 1
                print(f"  ⏱️ Tiempo: {processing_time:.1f} segundos")
                print(f"  ✅ REGRIDDING EXITOSO")
                
            except Exception as e:
                # Registrar error
                processing_time = time.time() - start_time
                total_processing_time += processing_time
                
                results[var][scenario] = {
                    'status': 'failed',
                    'error': str(e),
                    'processing_time': processing_time,
                    'input_file': input_file
                }
                
                processing_log.append({
                    'combination': combination,
                    'status': 'FAILED',
                    'processing_time': processing_time,
                    'error': str(e)
                })
                
                failed += 1
                print(f"  ❌ ERROR: {e}")
                print(f"  ⏱️ Tiempo hasta error: {processing_time:.1f} segundos")
                
                # Continuar con el siguiente archivo
                continue
    
    # Resumen final
    print(f"\n{'='*70}")
    print(f"📊 RESUMEN FINAL - REGRIDDING MASIVO")
    print(f"{'='*70}")
    print(f"✅ Exitosos: {successful}/{total_files}")
    print(f"❌ Fallidos: {failed}/{total_files}")
    print(f"⏱️ Tiempo total: {total_processing_time:.1f} segundos ({total_processing_time/60:.1f} minutos)")
    print(f"📁 Directorio salida: {regridded_dir}")
    
    if successful > 0:
        print(f"\n📋 ARCHIVOS REGRIDDED GENERADOS:")
        total_output_size = 0
        successful_coverage = 0
        
        for log in processing_log:
            if log['status'] == 'SUCCESS':
                coverage_emoji = "✅" if log['valle_coverage'] else "⚠️"
                compression = log['compression_ratio']
                print(f"  {coverage_emoji} {log['output_file']} ({log['output_size_mb']:.1f} MB, {compression:.1f}x comp)")
                total_output_size += log['output_size_mb']
                if log['valle_coverage']:
                    successful_coverage += 1
        
        print(f"  📊 Tamaño total: {total_output_size:.1f} MB")
        print(f"  🎯 Cobertura Valle: {successful_coverage}/{successful} archivos")
        
        if successful == total_files:
            print(f"\n🎉 REGRIDDING COMPLETADO EXITOSAMENTE")
            print(f"🎯 Todos los archivos procesados y listos")
            print(f"📋 Próximos pasos posibles:")
            print(f"  1. 🔍 Verificar calidad de regridding")
            print(f"  2. 📊 Análisis espacial/temporal")
            print(f"  3. 🔧 Bias correction (cuando estén disponibles los parámetros)")
        else:
            print(f"\n⚠️ REGRIDDING PARCIALMENTE COMPLETADO")
            print(f"📋 Revisa los errores y reintenta archivos fallidos")
    
    if failed > 0:
        print(f"\n❌ ERRORES ENCONTRADOS:")
        for log in processing_log:
            if log['status'] == 'FAILED':
                print(f"  ❌ {log['combination']}: {log.get('error', 'Error desconocido')}")
    
    # Guardar log detallado
    log_filename = f"ssp_regridding_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    log_path = logs_dir / log_filename
    
    with open(log_path, 'w') as f:
        json.dump(processing_log, f, indent=2, default=str)
    
    print(f"\n📝 Log detallado guardado: {log_path}")
    print(f"🏁 Pipeline de regridding completado")
    print(f"{'='*70}")
    
    return results, processing_log


# Ejecutar pipeline de regridding si todo está listo
if ('can_regrid' in locals() and can_regrid and 
    'target_grids' in locals() and target_grids is not None and
    'ssp_files' in locals() and ssp_files):
    
    print(f"\n🚀 INICIANDO PIPELINE DE REGRIDDING MASIVO")
    print(f"📝 Procesando {len(variables)} variables × {len(ssp_scenarios)} escenarios")
    print(f"🎯 Destino: Resolución CR2MET (0.05°) en Valle de Aconcagua")
    print(f"🔧 CORRECCIONES APLICADAS:")
    print(f"   ✅ Coordenadas guardadas como variables explícitas")
    print(f"   ✅ Encoding mejorado para NetCDF")
    print(f"   ✅ Atributos boolean convertidos a enteros")
    
    # Primero ejecutar test de la función corregida
    print(f"\n🧪 EJECUTANDO TEST DE REGRIDDING CORREGIDO...")
    test_result = test_regridding_function()
    
    if test_result:
        print(f"\n✅ Test exitoso - Procediendo con procesamiento masivo")
        # Ejecutar procesamiento
        regridding_results, regridding_log = process_ssp_regridding_pipeline()
        
        print(f"\n🎯 PIPELINE DE REGRIDDING COMPLETADO")
        print(f"📁 Archivos regridded disponibles en: {regridded_dir}")
        print(f"📋 Listos para análisis o bias correction futuro")
    else:
        print(f"\n❌ Test de regridding falló - No se ejecutará procesamiento masivo")
        print(f"📋 Revisa la función de regridding y vuelve a intentar")
else:
    print(f"\n❌ No se puede ejecutar regridding. Revisa prerrequisitos:")
    print(f"  - Archivos SSP: {'✅' if 'ssp_files' in locals() and ssp_files else '❌'}")
    print(f"  - CR2MET: {'✅' if 'cr2met_ref' in locals() and cr2met_ref is not None else '❌'}")
    print(f"  - Rejilla objetivo: {'✅' if 'target_grids' in locals() and target_grids is not None else '❌'}")
    print(f"  - Variable can_regrid: {'✅' if 'can_regrid' in locals() and can_regrid else '❌'}")
    print(f"\n💡 Ejecuta las celdas anteriores en orden para definir todas las variables necesarias.")


🚀 INICIANDO PIPELINE DE REGRIDDING MASIVO
📝 Procesando 3 variables × 3 escenarios
🎯 Destino: Resolución CR2MET (0.05°) en Valle de Aconcagua
🔧 CORRECCIONES APLICADAS:
   ✅ Coordenadas guardadas como variables explícitas
   ✅ Encoding mejorado para NetCDF
   ✅ Atributos boolean convertidos a enteros

🧪 EJECUTANDO TEST DE REGRIDDING CORREGIDO...

🧪 TEST DE FUNCIÓN DE REGRIDDING CORREGIDA
📄 Archivo de test: pr_ACCESS-CM2_ssp245_concatenated_coords_2015-2100.nc
🔬 Variable: pr, Escenario: ssp245
📊 Sample shape: {'time': 10, 'bnds': 2, 'lat': 144, 'lon': 192}
  🌍 Iniciando regridding...
    Método: bilinear
    Motor: xESMF
    📊 Origen: 144 × 192 puntos
    📊 Destino: 24 × 42 puntos
    📈 Variable: pr
    🔧 Usando xESMF para regridding...
    ✅ Regridding xESMF exitoso
    📊 Resultado: {'time': 10, 'lat': 24, 'lon': 42}
    📈 Variable: pr
    🗂️ Coordenadas: ['time', 'lat', 'lon']
    ✅ Coordenadas como variables: ['time', 'lat', 'lon']

✅ TEST EXITOSO:
  ⏱️ Tiempo: 1.5 segundos
  📊 Forma 

In [29]:
# ============================================================
# 🧪 CELDA DE TEST INDEPENDIENTE PARA VERIFICAR CORRECCIONES
# ============================================================

print("🧪 EJECUTAR TEST DE REGRIDDING CORREGIDO")
print("=" * 60)

# Verificar prerrequisitos básicos
prereqs_ok = True
if 'ssp_files' not in locals() or not ssp_files:
    print("❌ Archivos SSP no disponibles")
    prereqs_ok = False
    
if 'target_grids' not in locals() or target_grids is None:
    print("❌ Rejilla objetivo no disponible")  
    prereqs_ok = False

if prereqs_ok:
    print("✅ Prerrequisitos cumplidos - Ejecutando test...")
    test_success = test_regridding_function()
    
    if test_success:
        print("\n🎯 RESULTADO DEL TEST:")
        print("✅ La función de regridding CORREGIDA funciona correctamente")
        print("✅ Las coordenadas se guardan como variables explícitas")
        print("✅ El archivo NetCDF resultante tiene estructura correcta")
        print("\n🚀 Puedes proceder con el procesamiento masivo ejecutando la celda anterior")
    else:
        print("\n❌ RESULTADO DEL TEST:")
        print("❌ La función de regridding aún tiene problemas")
        print("📋 Revisa la implementación antes de proceder")
else:
    print("\n💡 PARA EJECUTAR EL TEST:")
    print("1. Asegúrate de haber ejecutado las celdas de carga de datos")
    print("2. Verifica que los archivos SSP concatenados están disponibles")
    print("3. Confirma que la rejilla objetivo de CR2MET se creó correctamente")
    print("4. Vuelve a ejecutar esta celda")

🧪 EJECUTAR TEST DE REGRIDDING CORREGIDO
✅ Prerrequisitos cumplidos - Ejecutando test...

🧪 TEST DE FUNCIÓN DE REGRIDDING CORREGIDA
📄 Archivo de test: pr_ACCESS-CM2_ssp245_concatenated_coords_2015-2100.nc
🔬 Variable: pr, Escenario: ssp245
📊 Sample shape: {'time': 10, 'bnds': 2, 'lat': 144, 'lon': 192}
  🌍 Iniciando regridding...
    Método: bilinear
    Motor: xESMF
    📊 Origen: 144 × 192 puntos
    📊 Destino: 24 × 42 puntos
    📈 Variable: pr
    🔧 Usando xESMF para regridding...
    ✅ Regridding xESMF exitoso
    📊 Resultado: {'time': 10, 'lat': 24, 'lon': 42}
    📈 Variable: pr
    🗂️ Coordenadas: ['time', 'lat', 'lon']
    ✅ Coordenadas como variables: ['time', 'lat', 'lon']

✅ TEST EXITOSO:
  ⏱️ Tiempo: 1.5 segundos
  📊 Forma original: 144 × 192
  📊 Forma regridded: 24 × 42
  📈 Reducción de puntos: 27.4x
  ✅ Coordenadas como variables: ['time', 'lat', 'lon']
  📏 Tamaño archivo de test: 0.0 MB
  ✅ Verificación: coordenadas guardadas como variables: ['time', 'lat', 'lon']
  🎯 TEST C

In [30]:
# ============================================================
# 🔍 VERIFICACIÓN DE CALIDAD DE ARCHIVOS REGRIDDED
# ============================================================

def verify_regridded_files_quality():
    """Verificar la calidad de los archivos regridded generados"""
    
    print("🔍 VERIFICACIÓN DE CALIDAD DE ARCHIVOS REGRIDDED")
    print("=" * 60)
    
    # Buscar archivos regridded
    regridded_files = list(regridded_dir.glob("*_regridded_cr2met_*.nc"))
    
    print(f"📦 Archivos encontrados: {len(regridded_files)}")
    
    if not regridded_files:
        print("❌ No se encontraron archivos regridded")
        return
    
    verification_results = {}
    
    for file_path in sorted(regridded_files):
        file_name = file_path.name
        file_size_mb = file_path.stat().st_size / (1024 * 1024)
        
        print(f"\n" + "="*50)
        print(f"📄 ARCHIVO: {file_name}")
        print(f"📊 Tamaño: {file_size_mb:.1f} MB")
        
        try:
            with xr.open_dataset(file_path) as ds:
                print(f"✅ Archivo se puede abrir correctamente")
                print(f"📊 Dimensiones: {dict(ds.dims)}")
                print(f"📈 Variables: {list(ds.data_vars)}")
                print(f"🗂️ Coordenadas: {list(ds.coords)}")
                
                # Verificar coordenadas como variables
                coord_vars = [coord for coord in ds.coords if coord in ds.variables]
                missing_coords = [coord for coord in ['time', 'lat', 'lon'] if coord not in coord_vars]
                
                print(f"✅ Coordenadas como variables: {coord_vars}")
                if missing_coords:
                    print(f"❌ Coordenadas faltantes: {missing_coords}")
                
                # Verificar rangos espaciales
                if 'lat' in ds.coords and 'lon' in ds.coords:
                    lat_range = f"{float(ds.lat.min()):.3f} a {float(ds.lat.max()):.3f}"
                    lon_range = f"{float(ds.lon.min()):.3f} a {float(ds.lon.max()):.3f}"
                    
                    print(f"🌍 Latitudes: {lat_range}")
                    print(f"🌍 Longitudes: {lon_range}")
                    
                    # Verificar cobertura Valle de Aconcagua
                    valle_lat_ok = (ds.lat.min().values <= valle_aconcagua_bounds['lat_min'] and 
                                   ds.lat.max().values >= valle_aconcagua_bounds['lat_max'])
                    valle_lon_ok = (ds.lon.min().values <= valle_aconcagua_bounds['lon_min'] and 
                                   ds.lon.max().values >= valle_aconcagua_bounds['lon_max'])
                    
                    valle_coverage = "✅" if (valle_lat_ok and valle_lon_ok) else "⚠️"
                    print(f"🎯 Cobertura Valle de Aconcagua: {valle_coverage}")
                
                # Verificar datos temporales
                if 'time' in ds.coords:
                    time_start = pd.to_datetime(ds.time.min().values)
                    time_end = pd.to_datetime(ds.time.max().values)
                    total_days = len(ds.time)
                    
                    print(f"📅 Período: {time_start.strftime('%Y-%m-%d')} a {time_end.strftime('%Y-%m-%d')}")
                    print(f"📈 Total días: {total_days:,}")
                    
                    # Verificar que cubre todo el período SSP
                    expected_days = (time_end - time_start).days + 1
                    completeness = (total_days / expected_days) * 100
                    print(f"📊 Completitud temporal: {completeness:.1f}%")
                
                # Verificar variable principal y sus datos
                main_vars = [var for var in ds.data_vars if var not in ['time_bnds', 'height']]
                if main_vars:
                    main_var = main_vars[0]
                    var_data = ds[main_var]
                    
                    print(f"📈 Variable principal: {main_var}")
                    print(f"📊 Forma: {var_data.shape}")
                    print(f"🔢 Tipo: {var_data.dtype}")
                    
                    # Tomar muestra para verificar datos
                    sample = var_data.isel(time=slice(0, min(30, len(ds.time))))
                    sample_vals = sample.values
                    
                    n_total = sample_vals.size
                    n_valid = np.sum(~np.isnan(sample_vals))
                    n_zeros = np.sum(sample_vals == 0)
                    
                    print(f"📊 Muestra (30 días):")
                    print(f"   Valores válidos: {n_valid:,} ({100*n_valid/n_total:.1f}%)")
                    print(f"   Valores cero: {n_zeros:,} ({100*n_zeros/n_total:.1f}%)")
                    print(f"   Valores NaN: {n_total-n_valid:,} ({100*(n_total-n_valid)/n_total:.1f}%)")
                    
                    if n_valid > 0:
                        valid_vals = sample_vals[~np.isnan(sample_vals)]
                        print(f"   Rango: {valid_vals.min():.3f} a {valid_vals.max():.3f}")
                        print(f"   Media: {valid_vals.mean():.3f}")
                
                # Verificar metadatos importantes
                important_attrs = ['regridding_method', 'regridding_engine', 'target_resolution', 'model', 'scenario']
                missing_attrs = [attr for attr in important_attrs if attr not in ds.attrs]
                
                if missing_attrs:
                    print(f"⚠️ Metadatos faltantes: {missing_attrs}")
                else:
                    print(f"✅ Metadatos importantes presentes")
                    print(f"   Método: {ds.attrs.get('regridding_method', 'N/A')}")
                    print(f"   Motor: {ds.attrs.get('regridding_engine', 'N/A')}")
                    print(f"   Resolución: {ds.attrs.get('target_resolution', 'N/A')}")
                
                # Registrar resultados
                verification_results[file_name] = {
                    'status': 'success',
                    'size_mb': file_size_mb,
                    'dims': dict(ds.dims),
                    'coords_ok': len(missing_coords) == 0,
                    'valle_coverage': valle_lat_ok and valle_lon_ok if 'lat' in ds.coords else False,
                    'temporal_completeness': completeness if 'time' in ds.coords else 0,
                    'data_quality': n_valid / n_total if main_vars else 0,
                    'has_metadata': len(missing_attrs) == 0
                }
                
        except Exception as e:
            print(f"❌ Error al verificar archivo: {e}")
            verification_results[file_name] = {
                'status': 'error',
                'error': str(e),
                'size_mb': file_size_mb
            }
    
    # Resumen final
    print(f"\n" + "="*60)
    print(f"📊 RESUMEN DE VERIFICACIÓN")
    print(f"=" * 60)
    
    successful = sum(1 for r in verification_results.values() if r['status'] == 'success')
    total = len(verification_results)
    
    print(f"✅ Archivos válidos: {successful}/{total}")
    
    if successful > 0:
        sizes = [r['size_mb'] for r in verification_results.values() if r['status'] == 'success']
        coverages = [r['valle_coverage'] for r in verification_results.values() if r['status'] == 'success']
        qualities = [r['data_quality'] for r in verification_results.values() if r['status'] == 'success']
        
        print(f"📊 Tamaño promedio: {np.mean(sizes):.1f} MB (rango: {min(sizes):.1f}-{max(sizes):.1f})")
        print(f"🎯 Cobertura Valle: {sum(coverages)}/{len(coverages)} archivos")
        print(f"📈 Calidad promedio de datos: {np.mean(qualities)*100:.1f}%")
        
        # Verificar si todos los archivos esperados están presentes
        expected_files = []
        for var in variables:
            for scenario in ssp_scenarios:
                expected_files.append(f"{var}_ACCESS-CM2_{scenario}_regridded_cr2met_2015-2100.nc")
        
        found_files = list(verification_results.keys())
        missing_files = [f for f in expected_files if f not in found_files]
        
        if missing_files:
            print(f"⚠️ Archivos faltantes: {missing_files}")
        else:
            print(f"✅ Todos los archivos esperados están presentes")
    
    if successful == total:
        print(f"\n🎉 VERIFICACIÓN EXITOSA")
        print(f"🎯 Todos los archivos regridded están correctos y listos para bias correction")
    else:
        failed = total - successful
        print(f"\n⚠️ VERIFICACIÓN PARCIAL")
        print(f"❌ {failed} archivos con problemas")
    
    return verification_results

print("🔍 Función de verificación lista")
print("💡 Ejecutar: verify_regridded_files_quality()")

🔍 Función de verificación lista
💡 Ejecutar: verify_regridded_files_quality()


In [ ]:
# ============================================================
# 🚀 EJECUTAR VERIFICACIÓN DE ARCHIVOS REGRIDDED
# ============================================================

# Ejecutar verificación completa de archivos
verification_results = verify_regridded_files_quality()

# Guardar resultados de verificación
verification_log_path = logs_dir / f"regridded_verification_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

with open(verification_log_path, 'w') as f:
    json.dump(verification_results, f, indent=2, default=str)

print(f"\n📝 Resultados de verificación guardados en: {verification_log_path}")

# Mostrar estadísticas rápidas
successful_files = [k for k, v in verification_results.items() if v['status'] == 'success']
total_files = len(verification_results)

print(f"\n🎯 RESULTADO FINAL:")
print(f"✅ Archivos exitosos: {len(successful_files)}/{total_files}")

if len(successful_files) == total_files:
    print(f"🎉 ¡REGRIDDING COMPLETAMENTE EXITOSO!")
    print(f"📁 Todos los archivos están listos para el siguiente paso")
    print(f"📋 Tamaños de archivos apropiados (70-85 MB cada uno)")
    print(f"✅ Coordenadas correctamente guardadas como variables")
    print(f"🎯 Cobertura espacial del Valle de Aconcagua confirmada")
else:
    failed_files = [k for k, v in verification_results.items() if v['status'] != 'success']
    print(f"⚠️ Archivos con problemas: {failed_files}")

In [31]:
verify_regridded_files_quality()

🔍 VERIFICACIÓN DE CALIDAD DE ARCHIVOS REGRIDDED
📦 Archivos encontrados: 9

📄 ARCHIVO: pr_ACCESS-CM2_ssp245_regridded_cr2met_2015-2100.nc
📊 Tamaño: 83.2 MB
✅ Archivo se puede abrir correctamente
📊 Dimensiones: {'time': 31411, 'lat': 24, 'lon': 42}
📈 Variables: ['pr']
🗂️ Coordenadas: ['time', 'lat', 'lon']
✅ Coordenadas como variables: ['time', 'lat', 'lon']
🌍 Latitudes: -33.325 a -32.175
🌍 Longitudes: -71.975 a -69.925
🎯 Cobertura Valle de Aconcagua: ✅
📅 Período: 2015-01-01 a 2100-12-31
📈 Total días: 31,411
📊 Completitud temporal: 100.0%
📈 Variable principal: pr
📊 Forma: (31411, 24, 42)
🔢 Tipo: float32
📊 Muestra (30 días):
   Valores válidos: 30,240 (100.0%)
   Valores cero: 0 (0.0%)
   Valores NaN: 0 (0.0%)
   Rango: 0.000 a 5.788
   Media: 0.093
✅ Metadatos importantes presentes
   Método: bilinear
   Motor: xESMF
   Resolución: 24x42

📄 ARCHIVO: pr_ACCESS-CM2_ssp370_regridded_cr2met_2015-2100.nc
📊 Tamaño: 83.2 MB
✅ Archivo se puede abrir correctamente
📊 Dimensiones: {'time': 31411, '

{'pr_ACCESS-CM2_ssp245_regridded_cr2met_2015-2100.nc': {'status': 'success',
  'size_mb': 83.1512861251831,
  'dims': {'time': 31411, 'lat': 24, 'lon': 42},
  'coords_ok': True,
  'valle_coverage': np.True_,
  'temporal_completeness': 100.0,
  'data_quality': np.float64(1.0),
  'has_metadata': True},
 'pr_ACCESS-CM2_ssp370_regridded_cr2met_2015-2100.nc': {'status': 'success',
  'size_mb': 83.18684101104736,
  'dims': {'time': 31411, 'lat': 24, 'lon': 42},
  'coords_ok': True,
  'valle_coverage': np.True_,
  'temporal_completeness': 100.0,
  'data_quality': np.float64(1.0),
  'has_metadata': True},
 'pr_ACCESS-CM2_ssp585_regridded_cr2met_2015-2100.nc': {'status': 'success',
  'size_mb': 83.1505880355835,
  'dims': {'time': 31411, 'lat': 24, 'lon': 42},
  'coords_ok': True,
  'valle_coverage': np.True_,
  'temporal_completeness': 100.0,
  'data_quality': np.float64(1.0),
  'has_metadata': True},
 'tasmax_ACCESS-CM2_ssp245_regridded_cr2met_2015-2100.nc': {'status': 'success',
  'size_mb':